# 03 · Gobernanza — quién puede ver qué

**Pregunta que se contesta aquí:** ¿qué equipo debería poder leer qué capa de datos, y por qué?

## Qué pasa en este notebook

Hasta ahora el proyecto construyó datos; este notebook decide **quién puede acceder a ellos**. Unity Catalog organiza todo en tres niveles (`catálogo.schema.tabla`) y controla el acceso con permisos jerárquicos: para leer una tabla hace falta tener los tres privilegios a la vez, `USE CATALOG`, `USE SCHEMA` y `SELECT`. Los dos primeros solo permiten atravesar la jerarquía; únicamente `SELECT` da acceso a los datos.

El principio que guía las decisiones es el de **mínimo privilegio**: cada equipo recibe solo lo que necesita para hacer su trabajo.

**Decisión documentada:** el equipo de BI y analítica trabaja con métricas ya agregadas, así que recibe lectura sobre el esquema `gold` y **nada** sobre `bronze`. Bronze es el dato crudo: contiene el nombre y el correo de los clientes, y también filas con errores que llevarían a conclusiones equivocadas. Gold no tiene datos personales y ya pasó por la limpieza.

**En esta celda** se hereda la configuración compartida.

In [ ]:
%run ./00_Setup/00_Configuracion

**En esta celda** se importan las funciones de PySpark.

In [ ]:
from pyspark.sql import functions as F

**En esta celda** se comprueba con datos (y no de memoria) que las tablas del proyecto son **managed**: Unity Catalog es dueño de los metadatos y de los archivos físicos. `information_schema.tables` expone `table_type` (`MANAGED` o `EXTERNAL`) para todo el catálogo. Es la elección correcta aquí porque los datos viven solo en el Lakehouse y ningún otro sistema necesita leer los archivos directamente.

In [ ]:
display(spark.sql(f"""
    SELECT table_schema, table_name, table_type
    FROM {CATALOG_NAME}.information_schema.tables
    WHERE table_schema IN ('{SCHEMA_BRONZE}', '{SCHEMA_SILVER}', '{SCHEMA_GOLD}')
    ORDER BY table_schema, table_name
"""))

**En esta celda** se busca en qué esquemas existen las columnas con datos personales (`email` y `nombre`). Es la evidencia que justifica la decisión de gobernanza: solo aparecen en Bronze, así que dar acceso a Gold no expone a ningún cliente.

In [ ]:
display(spark.sql(f"""
    SELECT table_schema, table_name, column_name
    FROM {CATALOG_NAME}.information_schema.columns
    WHERE table_schema IN ('{SCHEMA_BRONZE}', '{SCHEMA_SILVER}', '{SCHEMA_GOLD}')
      AND column_name IN ('email', 'nombre')
    ORDER BY table_schema, table_name
"""))

**En esta celda** se otorgan los tres privilegios necesarios para leer Gold: `USE CATALOG`, `USE SCHEMA` sobre `gold` y `SELECT` sobre `gold`. No se otorga nada sobre Bronze ni Silver.

**Nota sobre Free Edition:** el workspace es de un solo usuario y no tiene grupos, así que los `GRANT` se aplican sobre `current_user()` como reemplazo (*stand-in*) de un grupo real. La sintaxis es idéntica a la de un workspace de equipo; solo cambia el nombre del principal (por ejemplo `analistas_bi`). Se otorga sobre el **esquema** y no sobre tablas sueltas, para que las tablas Gold que se creen en el futuro queden cubiertas automáticamente.

In [ ]:
usuario_actual = spark.sql("SELECT current_user() AS u").first()["u"]
print(f"Principal actual (stand-in del grupo de BI): {usuario_actual}")

for stmt in [
    f"GRANT USE CATALOG ON CATALOG {CATALOG_NAME} TO `{usuario_actual}`",
    f"GRANT USE SCHEMA ON SCHEMA {CATALOG_NAME}.{SCHEMA_GOLD} TO `{usuario_actual}`",
    f"GRANT SELECT ON SCHEMA {CATALOG_NAME}.{SCHEMA_GOLD} TO `{usuario_actual}`",
]:
    spark.sql(stmt)
    print(f"[OK] {stmt}")

print("\nEn un workspace de equipo, con grupos configurados, seria:")
print(f"  GRANT SELECT ON SCHEMA {CATALOG_NAME}.{SCHEMA_GOLD} TO `analistas_bi`")
print(f"  (y ningun permiso para analistas_bi sobre {CATALOG_NAME}.{SCHEMA_BRONZE})")

**En esta celda** se audita lo otorgado con `SHOW GRANTS`, que devuelve una fila por cada combinación de principal y privilegio. Es la verificación de que el permiso quedó como se documentó.

In [ ]:
print(f"Permisos sobre {CATALOG_NAME}.{SCHEMA_GOLD}:")
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOG_NAME}.{SCHEMA_GOLD}"))
print(f"Permisos sobre el catalogo {CATALOG_NAME}:")
display(spark.sql(f"SHOW GRANTS ON CATALOG {CATALOG_NAME}"))

Gobernanza · `GRANT` y mínimo privilegio &nbsp;·&nbsp; Siguiente: **04_Orquestacion**